# Forecasting 3 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Sportswear retail

**Problem:** Forecast next-week store sales for staffing and stock planning.

**Proposed success criterion:** Beat a four-week rolling-mean baseline MAE by at least 10% on the last 12 weeks; compare stores.

**Dataset:** `forecasting_3.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [2]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_3.csv')
if not DATA.exists(): DATA = Path('datasets/forecasting_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(841, 12)


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur,list_price_eur
0,S01,01/01/2024,Yes,0,464,13.5,579,Large,14.81,242.75,13098.18,17.18
1,S01,08/01/2024,Yes,0,453,5.0,579,Large,15.25,262.27,14986.99,17.69
2,S01,15/01/2024,Yes,0,445,13.9,579,Large,14.48,485.51,13223.24,16.80
3,S01,22/01/2024,Yes,0,446,16.7,579,Large,14.73,398.76,15302.44,17.09
4,S01,29/01/2024,No,0,395,7.1,579,Large,13.73,180.64,13367.07,15.93


store_id                  0
week_start                0
promo_label               0
holiday_week              0
footfall_actual           0
rain_mm_actual            0
store_floor_m2            0
store_size_category       0
average_price_eur         0
returns_eur_after_week    0
sales_eur                 0
list_price_eur            0
dtype: int64

Exact duplicates: 1


This forecast supports the store manager and regional planner, who use it every week to decide staffing levels and how much stock to request for the coming week. The prediction moment is the end of the current week, before next week starts, so only information already known by then can be used as input. The observation unit is one store in one week, identified by store_id and week_start, and the target is sales_eur.

The proposed success criterion, beating a four week rolling mean baseline by 10 percent MAE, is measurable but incomplete on its own. A single overall MAE can hide the fact that some stores are harder to forecast than others, and MAE treats overestimating and underestimating demand as equally costly, which they are probably not for staffing and stock decisions. We propose keeping the MAE comparison as the main technical measure, but reporting it per store rather than only as one combined number, and separately tracking the share of weeks where demand is underestimated versus overestimated.

Two evidenced findings: the dataset is a complete, balanced weekly panel of 8 stores over 105 weeks each, with no missing values, so coverage is not a concern. We also found one exact duplicate row (store S02, week of 20 May 2024) and one implausible sales value of 82,000 euros for store S03, far above the next highest value of about 18,250 euros, both of which need to be addressed before modelling.

Two limitations: the returns_eur_after_week field is only known after the week is over, so it cannot be used as a predictor without leaking future information into the forecast. Also, promo_label contains four different spellings for what appears to be only two real categories, so the true promotional effect cannot be measured until these labels are cleaned.

## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [4]:
# Add your preparation code below. Inspect before/after evidence.
# Make a copy first:
work = df.copy()

work['week_start'] = pd.to_datetime(work['week_start'], dayfirst=True)

before = work.shape[0]
work = work.drop_duplicates()
print('Exact duplicates removed:', before - work.shape[0])

print('Before:', work['promo_label'].value_counts())
work['promo_flag'] = work['promo_label'].replace({'Yes': 1, 'promo': 1, 'No': 0, 'REGULAR': 0})
print('After:', work['promo_flag'].value_counts())

print('Correlation:', work[['average_price_eur', 'list_price_eur']].corr().iloc[0, 1])
work = work.drop(columns=['list_price_eur'])

work = work.drop(columns=['returns_eur_after_week'])

store_median = work.groupby('store_id')['sales_eur'].transform('median')
implausible = work['sales_eur'] > 3 * store_median
print('Implausible rows capped:', implausible.sum())
work.loc[implausible, 'sales_eur'] = store_median[implausible]

Exact duplicates removed: 1
Before: promo_label
No         638
Yes        196
REGULAR      4
promo        2
Name: count, dtype: int64
After: promo_flag
0    642
1    198
Name: count, dtype: int64
Correlation: 0.9999904317803365
Implausible rows capped: 1


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
|Exact duplicate row (S02, 2024-05-20)|Dropped the duplicate record|Row count 841 to 840|Counting it twice would double that store-week in training|Proceed with the deduplicated panel|
|promo_label had four inconsistent labels|Mapped to a clean binary promo_flag|638/196/4/2 before, clean 642/198 split after|Without this the promotion effect cannot be measured reliably|Use promo_flag as the predictor|
|average_price_eur and list_price_eur correlate at 0.9999|Dropped list_price_eur, kept average_price_eur|Correlation of 0.99999 before, single price field after|Keeping both adds no information and risks instability in linear models|Use average_price_eur only|

We also excluded returns_eur_after_week entirely, since it is only known after the week ends and using it would leak future information into the forecast, and we capped one implausible sales value for store S03 that was roughly four times higher than any other observation for that store.

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

The prediction moment is the end of the current week, before next week begins. At that point we know each store's characteristics (store_id, store_size_category, store_floor_m2), the calendar (holiday_week, month), whether a promotion is planned (promo_flag), the planned average price (average_price_eur), and the store's own sales history up to that point. We do not yet know footfall_actual or rain_mm_actual for the coming week, since both are only measured during the week itself, at the same time as sales_eur, so we exclude them from the model even though footfall_actual correlates with sales at 0.35. Including either would leak information from the future into a forecast that has to be made in advance.

We use a chronological split consistent with the forecasting task: for each store, the earliest weeks, once there is enough history to compute a four week rolling average, form the training set, and the last 12 weeks form the test set, matching the horizon in the proposed success criterion. All lag and rolling average features are built using only past weeks, shifted by one, so no future sales value is ever used to predict itself.

In [5]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

model_df = work.sort_values(['store_id', 'week_start']).copy()
model_df['month'] = model_df['week_start'].dt.month
model_df['sales_lag1'] = model_df.groupby('store_id')['sales_eur'].shift(1)
model_df['sales_roll4'] = model_df.groupby('store_id')['sales_eur'].shift(1).rolling(4).mean().reset_index(level=0, drop=True)
model_df = model_df.dropna(subset=['sales_lag1', 'sales_roll4']).reset_index(drop=True)

model_df['rank'] = model_df.groupby('store_id')['week_start'].rank(method='first', ascending=False)
test = model_df[model_df['rank'] <= 12].drop(columns='rank')
train = model_df[model_df['rank'] > 12].drop(columns='rank')
print('Train rows:', train.shape[0], 'Test rows:', test.shape[0])

target = 'sales_eur'
features = ['store_id', 'store_size_category', 'store_floor_m2', 'holiday_week',
            'promo_flag', 'average_price_eur', 'month', 'sales_lag1', 'sales_roll4']
categorical = ['store_id', 'store_size_category']

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

baseline_pred = test['sales_roll4']
baseline_mae = mean_absolute_error(y_test, baseline_pred)

pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical)], remainder='passthrough')
candidate = Pipeline([('pre', pre), ('model', RandomForestRegressor(n_estimators=300, random_state=42))])
candidate.fit(X_train, y_train)
candidate_pred = candidate.predict(X_test)
candidate_mae = mean_absolute_error(y_test, candidate_pred)

print('Baseline MAE (four-week rolling mean):', round(baseline_mae, 2))
print('Candidate MAE (Random Forest):', round(candidate_mae, 2))
print('Improvement over baseline:', round((1 - candidate_mae / baseline_mae) * 100, 1), '%')

test = test.copy()
test['baseline_err'] = abs(test['sales_eur'] - test['sales_roll4'])
test['candidate_err'] = abs(test['sales_eur'] - candidate_pred)
print(test.groupby('store_id')[['baseline_err', 'candidate_err']].mean())

Train rows: 711 Test rows: 96
Baseline MAE (four-week rolling mean): 790.48
Candidate MAE (Random Forest): 733.82
Improvement over baseline: 7.2 %
          baseline_err  candidate_err
store_id                             
S01        1192.243958    1070.757461
S02         677.298750     663.679403
S03         800.757917     717.739433
S04         810.048125     573.674525
S05         806.045833     942.949597
S06         685.113750     706.869169
S07         806.134375     676.656386
S08         546.161458     518.197603


## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.


The candidate model reduced the forecast error from 790.48 to 733.82 euros, a 7.2 percent improvement over the four week rolling mean baseline. This falls short of the 10 percent target proposed in Phase 1, so the success criterion was not met in full.

The improvement is not consistent across stores. Five of the eight stores improved meaningfully, store S04 in particular dropped from about 810 to 574 euros of average error, a close to 30 percent gain. Two stores, S05 and S06, actually got worse with the candidate model than with the simple rolling mean.

Looking at the direction of the error, not just its size, the model systematically underestimates sales in every single store, by about 310 euros per week on average, and underestimates in 60 percent of weeks against 40 percent overestimated. This is the more costly direction for this business, since Phase 1 identified understaffing and stockouts as more damaging than overstaffing, so this bias is a real concern, not just a technical detail.

We do not have cost data for staffing or stock, so we cannot yet translate the error reduction into actual money saved. The preparation decisions from Phase 3 did help: removing the leaking fields and the duplicate row, and cleaning the promo labels, produced a model that beats the baseline without relying on information that would not actually be available at forecast time. Given the mixed store level results, the missed target, and the consistent underestimation bias, we recommend returning to Phase 4 before deployment, trying a per store model, a longer rolling window, or a correction for the underestimation bias, rather than treating the current single pooled model as final.

In [7]:
test['signed_error'] = test['sales_eur'] - candidate_pred
print(test.groupby('store_id')['signed_error'].mean().round(1))
print()
print('Overall mean signed error:', round(test['signed_error'].mean(), 1))
print('Share of weeks underestimated:', round((test['signed_error'] > 0).mean(), 2))
print('Share of weeks overestimated:', round((test['signed_error'] < 0).mean(), 2))

store_id
S01    429.2
S02     18.3
S03    234.1
S04    393.0
S05    698.8
S06    120.0
S07    310.1
S08    267.6
Name: signed_error, dtype: float64

Overall mean signed error: 308.9
Share of weeks underestimated: 0.6
Share of weeks overestimated: 0.4


## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

The store manager and regional planner use this forecast every week, generated each Sunday evening using all sales data through the end of the current week, in time to plan staffing and submit the stock replenishment order before the new week starts. The predicted demand per store feeds directly into two decisions: how many staff hours to schedule and how much stock to request from the warehouse. The store manager can always override the suggested numbers, for example when they know about a local event, a closure, or a planned promotion that the model was not trained on. We would track the weekly forecast error, MAE, per store, and compare it against the four week rolling mean baseline every week rather than just once. Given that store S05 and S06 performed worse with this model than with the simple baseline during evaluation, each store would start on the candidate model only after it beats its own store specific baseline for at least four consecutive weeks, otherwise that store stays on the rolling mean. If any store's weekly error exceeds its historical baseline error for three weeks in a row after deployment, the model is automatically paused for that store and reverted to the rolling mean until the team investigates. Since the model was found to systematically underestimate sales, the recommended stock and staffing numbers would include a small built in buffer rather than being used exactly as predicted. No individual customer data is used anywhere in this process, only store level weekly aggregates, which limits privacy risk, but store managers should still be told clearly that the forecast is a planning aid and not a guarantee, so staffing decisions are not made rigidly against it.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

Group: Controlled pilot, not a full rollout.

The candidate model beats the four week rolling mean baseline in six of the eight stores, sometimes substantially, for example almost 30 percent lower error in store S04. We recommend starting with a pilot in those six stores only, keeping S05 and S06 on the simple rolling mean baseline until we understand why the model performs worse there. We also recommend building in the small forecast buffer described in Phase 6 from the start, since the model systematically underestimates demand across every store. This stops short of a full deployment because the 10 percent accuracy target from Phase 1 was not met overall, and because we still lack cost data to confirm that the error reduction translates into real savings in staffing or stock.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.

Daniel: One preparation choice I pushed for was dropping list_price_eur instead of average_price_eur, since the two were correlated at 0.9999 and keeping both would have added nothing but redundancy. One finding that stood out to me was that the candidate model consistently underestimates sales rather than making random errors in both directions, which is the more expensive mistake for a store trying to avoid stockouts. The business implication is that even a model with a lower average error is not automatically safe to deploy everywhere, since two of our eight stores actually got worse results than the simple baseline, which is why I think a full rollout would have been premature.